# Metrics & Outputs

Analysis of simulation results — ordered from most operationally critical to most granular.

| Section | Key question |
|---|---|
| **Foregone Revenue** | How much revenue is lost to care gaps, and where? |
| **Loss-to-Follow-Up** | Where do patients drop out of the care pathway? |
| **Cervical Screening Pipeline** | What fraction of eligible patients complete each clinical step? |
| **Lung Cancer Pathway** | How many eligible patients reach diagnosis and treatment? |
| **Cervical Results by Age** | Are result distributions matching what we'd expect clinically? |
| **CIN Grade Distribution** | What procedure mix should we expect per colposcopy referral? |

> Revenue figures use placeholder CPT rates. Replace with NYP contract rates before financial planning.

In [ ]:
import sys
sys.path.insert(0, '../src')

import config as cfg
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from metrics import compute_rates, compute_revenue, print_revenue_summary

## Setup — Run Simulation

Run the cell below to execute a fresh 1-year simulation, or skip it if `metrics` is already in scope from `simulation.ipynb`.

In [ ]:
# Run a fresh 1-year simulation if metrics are not already in scope.
# If you ran simulation.ipynb first, comment this cell out — metrics will already be defined.
import sys
sys.path.insert(0, '../src')

import config as cfg
from runner import SimulationRunner
from metrics import compute_rates, compute_revenue, print_revenue_summary

sim = SimulationRunner(n_days=365, seed=cfg.RANDOM_SEED)
metrics = sim.run()
print(f"Simulation complete — {metrics['n_patients']:,} patients processed.")

---

## Foregone Revenue & Screening Capacity

Revenue lost at each point where a patient drops out of the care pathway — and the procedures that were actually completed. The capture rate tells you what share of potential revenue the program is realising today.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from metrics import compute_revenue, print_revenue_summary

rev = compute_revenue(metrics)

real_items = {k: v for k, v in rev['realized_by_procedure'].items() if v > 0}
fore_items = {k: v for k, v in rev['foregone_by_node'].items()      if v > 0}

total_addressable = rev['realized_total'] + rev['foregone_total']
capture_pct = 100 * rev['realized_total'] / max(total_addressable, 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Left: realized revenue by procedure ───────────────────────────────────────
ax = axes[0]
if real_items:
    lbls = [k.replace('_', '\n') for k in real_items]
    amts = list(real_items.values())
    bars = ax.bar(lbls, [a / 1000 for a in amts], color='#1565C0', alpha=0.88)
    ax.set_ylabel('Revenue ($k)', fontsize=10)
    ax.set_title(f'Realized Revenue by Procedure\nTotal: ${rev["realized_total"]:,.0f}',
                 fontsize=11, fontweight='bold')
    for bar, amt in zip(bars, amts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'${amt/1000:.1f}k', ha='center', fontsize=8)
    plt.setp(ax.get_xticklabels(), fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}k'))

# ── Right: foregone revenue by LTFU node ──────────────────────────────────────
ax = axes[1]
if fore_items:
    lbls2 = [k.replace('_', '\n') for k in fore_items]
    amts2 = list(fore_items.values())
    bars2 = ax.bar(lbls2, [a / 1000 for a in amts2], color='#C62828', alpha=0.88)
    ax.set_ylabel('Revenue foregone ($k)', fontsize=10)
    ax.set_title(f'Foregone Revenue by LTFU Node\nTotal: ${rev["foregone_total"]:,.0f}',
                 fontsize=11, fontweight='bold')
    for bar, amt in zip(bars2, amts2):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'${amt/1000:.1f}k', ha='center', fontsize=8)
    plt.setp(ax.get_xticklabels(), fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}k'))
else:
    axes[1].text(0.5, 0.5, 'No foregone revenue recorded', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=11, color='gray')

plt.suptitle(
    f'Revenue Analysis  ·  Capture rate: {capture_pct:.1f}%  ·  '
    f'Foregone: {100 - capture_pct:.1f}%  ·  (PLACEHOLDER CPT rates)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

print_revenue_summary(metrics)

---

## Loss-to-Follow-Up

Where patients are dropping out — and how many at each point. The largest bar is the highest-priority target for care coordination.

In [ ]:
from metrics import compute_rates

rates = compute_rates(metrics)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: LTFU count by node ───────────────────────────────────────────────
ax = axes[0]
ltfu_nodes  = ['Post-abnormal\n(no colposcopy)', 'Post-colposcopy\n(no treatment)', 'Declined\nscreening']
ltfu_counts = [metrics['ltfu_post_abnormal'], metrics['ltfu_post_colposcopy'], metrics['ltfu_unscreened']]
ltfu_colors = ['#C62828', '#B71C1C', '#BDBDBD']
bars = ax.bar(ltfu_nodes, ltfu_counts, color=ltfu_colors)
ax.set_ylabel('Patients lost at node')
ax.set_title('LTFU Count by Node', fontsize=11, fontweight='bold')
for bar, cnt in zip(bars, ltfu_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{cnt:,}', ha='center', fontsize=9)

# ── Panel 2: Patient outcome pie ──────────────────────────────────────────────
ax = axes[1]
outcome_labels = ['Treated\n(LEEP/surgery)', 'LTFU', 'Untreated\n(declined tx)']
outcome_counts = [metrics['n_treated'], metrics['n_ltfu'], metrics['n_untreated']]
outcome_colors = ['#388E3C', '#C62828', '#F57C00']
nonzero = [(l, c, col) for l, c, col in zip(outcome_labels, outcome_counts, outcome_colors) if c > 0]
if nonzero:
    lbls, cnts, cols = zip(*nonzero)
    ax.pie(cnts, labels=lbls, colors=cols, autopct='%1.1f%%', startangle=90,
           textprops={'fontsize': 9})
    ax.set_title('Patient Outcomes\n(all exited patients)', fontsize=11, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No exited patients yet', ha='center', va='center',
            transform=ax.transAxes, fontsize=10, color='gray')
    ax.set_title('Patient Outcomes')

# ── Panel 3: Key performance rates ────────────────────────────────────────────
ax = axes[2]
rate_labels = [
    'Cervical\nscreening rate',
    'Abnormal\nresult rate',
    'Colposcopy\ncompletion\n(of abnormals)',
    'Excisional tx\n(of colposcopies)',
]
rate_values = [
    rates['screening_rate_cervical_pct'],
    rates['abnormal_rate_cervical_pct'],
    rates['colposcopy_completion_pct'],
    rates['treatment_completion_pct'],
]
rate_colors = ['#1565C0', '#F57C00', '#6A1B9A', '#1B5E20']
bars2 = ax.bar(rate_labels, rate_values, color=rate_colors, alpha=0.85)
ax.set_ylabel('Rate (%)')
ax.set_title('Key Performance Rates', fontsize=11, fontweight='bold')
ax.set_ylim(0, 115)
for bar, val in zip(bars2, rate_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontsize=9)

plt.suptitle('Loss-to-Follow-Up & Outcome Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Cervical Screening Pipeline

Of everyone seen by a provider, how many were eligible, how many got screened, and how many made it all the way to treatment? Each gap in the funnel is a missed care opportunity.

In [ ]:
_NORMAL_CERVICAL = {"NORMAL", "HPV_NEGATIVE"}
total_abnormal = sum(
    v for k, v in metrics['cervical_results'].items()
    if k not in _NORMAL_CERVICAL
)
cerv_excisional = (
    metrics['n_treatment'].get('leep', 0)
    + metrics['n_treatment'].get('cone_biopsy', 0)
)

funnel_stages = [
    ('Patients seen',                     metrics['n_patients']),
    ('Eligible (≥1 screen)',              metrics['n_eligible_any']),
    ('Cervical screened',                 metrics['n_screened']['cervical']),
    ('Abnormal result',                   total_abnormal),
    ('Colposcopy completed',              metrics['n_colposcopy']),
    ('Excisional treatment\n(LEEP/cone)', cerv_excisional),
]

labels  = [s[0] for s in funnel_stages]
counts  = [s[1] for s in funnel_stages]
n_start = max(counts[0], 1)
pcts    = [c / n_start * 100 for c in counts]
colors  = ['#1565C0', '#1976D2', '#1E88E5', '#F57C00', '#E53935', '#43A047']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(labels[::-1], pcts[::-1], color=colors[::-1])
ax.set_xlabel('% of patients seen by a provider')
ax.set_title('Cervical Screening Pipeline — Funnel View', fontsize=13, fontweight='bold')
ax.set_xlim(0, 120)

for bar, pct, cnt in zip(bars, pcts[::-1], counts[::-1]):
    ax.text(pct + 1, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%  (n={cnt:,})', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---

## Lung Cancer Pathway

Of all patients eligible for lung cancer screening, how many made it through each step — from scan to biopsy to confirmed diagnosis to treatment. Each drop is a patient who didn't complete the next required step.

In [ ]:
if metrics['lung_eligible'] > 0:
    lung_steps = [
        ('Eligible (50–80, ≥20 pk-yrs)', 'lung_eligible'),
        ('LDCT order placed',             'lung_referral_placed'),
        ('LDCT scheduled',                'lung_ldct_scheduled'),
        ('LDCT completed',                'lung_ldct_completed'),
        ('Result communicated',           'lung_result_communicated'),
        ('Biopsy referral (RADS 4)',      'lung_biopsy_referral'),
        ('Biopsy scheduled',              'lung_biopsy_scheduled'),
        ('Biopsy completed',              'lung_biopsy_completed'),
        ('Malignancy confirmed',          'lung_malignancy_confirmed'),
        ('Treatment given',               'lung_treatment_given'),
    ]
    lung_steps = [(lbl, key) for lbl, key in lung_steps if metrics[key] > 0]
    lbls   = [s[0] for s in lung_steps]
    cnts   = [metrics[s[1]] for s in lung_steps]
    n_elig = max(cnts[0], 1)

    palette = list(plt.cm.Blues_r(range(40, 220, max(1, 180 // len(cnts)))))
    if len(cnts) >= 2: palette[-1] = (0.40, 0.73, 0.42, 1.0)
    for i, (_, key) in enumerate(lung_steps):
        if key == 'lung_malignancy_confirmed':
            palette[i] = (0.90, 0.33, 0.32, 1.0)

    fig, ax = plt.subplots(figsize=(12, max(4, len(cnts) * 0.65)))
    bars = ax.barh(lbls[::-1], [c / n_elig * 100 for c in cnts[::-1]], color=palette[::-1])
    ax.set_xlabel('% of eligible lung patients')
    ax.set_title('Lung LDCT Pathway Funnel', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 125)
    for bar, cnt in zip(bars, cnts[::-1]):
        pct = cnt / n_elig * 100
        ax.text(pct + 1, bar.get_y() + bar.get_height() / 2,
                f'{cnt:,}  ({pct:.1f}%)', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('No lung-eligible patients in this run.')

---

## Cervical Results by Age Group

How result categories are distributed across younger (21–29) and older (30–65) women. Younger women are screened by cytology only; older women may receive HPV-alone testing, which produces a different result distribution.

In [ ]:
strata_config = {
    'young':  ('young (21–29)',  'young'),
    'middle': ('middle (30–65)', 'middle_cytology'),
}
cyto_cats = ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL']
hpv_cats  = ['HPV_NEGATIVE', 'HPV_POSITIVE']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (stratum, (stratum_label, cfg_key)) in zip(axes[:2], strata_config.items()):
    sub   = metrics['cervical_by_age_stratum'].get(stratum, {})
    total = max(sum(sub.values()), 1)
    cats  = [c for c in cyto_cats if c in sub or cfg_key in cfg.CERVICAL_RESULT_PROBS]
    obs   = [sub.get(c, 0) / total * 100 for c in cats]
    exp   = [cfg.CERVICAL_RESULT_PROBS.get(cfg_key, {}).get(c, 0) * 100 for c in cats]
    x = range(len(cats))
    ax.bar([i - 0.2 for i in x], obs, width=0.4, label='Observed', color='#1565C0', alpha=0.85)
    ax.bar([i + 0.2 for i in x], exp, width=0.4, label='Expected', color='#90CAF9', alpha=0.85)
    ax.set_xticks(list(x)); ax.set_xticklabels(cats, rotation=30, ha='right')
    ax.set_ylabel('% of screenings')
    ax.set_title(f'Cytology — {stratum_label}\n(n={sum(sub.values()):,})')
    ax.legend(fontsize=8)

ax = axes[2]
middle_sub = metrics['cervical_by_age_stratum'].get('middle', {})
total_mid  = max(sum(middle_sub.values()), 1)
obs_hpv = [middle_sub.get(c, 0) / total_mid * 100 for c in hpv_cats]
exp_hpv = [cfg.CERVICAL_RESULT_PROBS['middle_hpv'].get(c, 0) * 100 for c in hpv_cats]
x = range(len(hpv_cats))
ax.bar([i - 0.2 for i in x], obs_hpv, width=0.4, label='Observed', color='#1565C0', alpha=0.85)
ax.bar([i + 0.2 for i in x], exp_hpv, width=0.4, label='Expected', color='#90CAF9', alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(hpv_cats, rotation=15, ha='right')
ax.set_ylabel('% of HPV-alone screenings')
ax.set_title('HPV-alone — middle (30–65)')
ax.legend(fontsize=8)

plt.suptitle('Cervical Result Distribution: Observed vs. Expected', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## CIN Grade Distribution

For patients referred to colposcopy, what mix of findings should we expect depending on the result that triggered the referral? Higher-grade triggers produce more CIN2/3, which means more treatment procedures per colposcopy performed.

In [ ]:
if metrics['n_colposcopy'] > 0:
    triggers   = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POSITIVE']
    cin_grades = ['NORMAL', 'CIN1', 'CIN2', 'CIN3']
    colors_cin = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']

    fig, ax = plt.subplots(figsize=(10, 5))
    bottoms = [0.0] * len(triggers)
    for i, grade in enumerate(cin_grades):
        vals = [cfg.COLPOSCOPY_RESULT_PROBS.get(f'from_{t}', {}).get(grade, 0) * 100
                for t in triggers]
        ax.bar(triggers, vals, bottom=bottoms, color=colors_cin[i], label=grade, alpha=0.9)
        bottoms = [b + v for b, v in zip(bottoms, vals)]

    ax.set_ylabel('CIN grade distribution (%)')
    ax.set_title('Expected CIN Grade Mix by Triggering Result\n(from config probability tables)',
                 fontsize=12, fontweight='bold')
    ax.legend(title='CIN grade', loc='upper right')
    ax.set_ylim(0, 110)
    ax.text(0.01, 0.97, f'Total colposcopies in run: {metrics["n_colposcopy"]:,}',
            transform=ax.transAxes, va='top', fontsize=9, color='#333')
    plt.tight_layout()
    plt.show()
else:
    print('No colposcopies performed in this run.')